## 1. Load data
We load the EV charging dataset and inspect the first rows.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("EVChargingStationUsage.csv", low_memory=False)

print("Shape:", df.shape)
display(df.head())


## 2. Basic quality checks
We check missing values, duplicates, and data types.

In [ ]:
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False))

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nData types:")
display(df.dtypes)

## 3. Clean text columns
We remove extra spaces and normalize station names.

In [ ]:
text_cols = ["Station Name", "Port Type", "Plug Type", "Ended By", "Start Time Zone", "End Time Zone"]

for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

df["Station Name"] = (
    df["Station Name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*#\s*", " #", regex=True)
)

df["Start Time Zone"] = df["Start Time Zone"].str.upper()
df["End Time Zone"] = df["End Time Zone"].str.upper()

## 4. Parse dates and durations
We convert date/time columns and duration strings into proper formats.

In [ ]:
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["End Date"] = pd.to_datetime(df["End Date"], errors="coerce")
df["Transaction Date (Pacific Time)"] = pd.to_datetime(df["Transaction Date (Pacific Time)"], errors="coerce")

total_duration = pd.to_timedelta(df["Total Duration (hh:mm:ss)"], errors="coerce")
charging_duration = pd.to_timedelta(df["Charging Time (hh:mm:ss)"], errors="coerce")

## 5. Convert numeric fields
We convert important numeric columns to numeric values.

In [ ]:
numeric_cols = ["Energy (kWh)", "Fee", "Latitude", "Longitude", "GHG Savings (kg)", "Gasoline Savings (gallons)"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## 6. Build cleaned session dataset
We remove exact duplicates and keep sessions with valid start/end timestamps.

In [ ]:
invalid_time = (
    df["Start Date"].isna() |
    df["End Date"].isna() |
    (df["End Date"] <= df["Start Date"])
)

timestamp_duration_min = (df["End Date"] - df["Start Date"]).dt.total_seconds() / 60
total_duration_min = total_duration.dt.total_seconds() / 60
duration_diff_min = (timestamp_duration_min - total_duration_min).abs()

print("Invalid time rows:", invalid_time.sum())
print("Rows with duration mismatch > 2 min:", (duration_diff_min > 2).sum())
print("Negative energy rows:", (df["Energy (kWh)"] < 0).sum())

df = df.drop_duplicates().copy()

clean_mask = (
    df["Start Date"].notna() &
    df["End Date"].notna() &
    (df["End Date"] > df["Start Date"]) &
    df["Station Name"].notna()
)

df_clean = df.loc[clean_mask].copy()

print("\nRows after duplicate removal:", len(df))
print("Rows in cleaned dataset:", len(df_clean))

## 7. Convert timestamps to UTC
We convert start/end times to UTC usind their timezone labels.

In [ ]:
zone_map = {"PST": "Etc/GMT+8", "PDT": "Etc/GMT+7", "UTC": "UTC"}

df_clean["Start_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")
df_clean["End_UTC"] = pd.Series(pd.NaT, index=df_clean.index, dtype="datetime64[ns, UTC]")

for label, tz in zone_map.items():
    m1 = (df_clean["Start Time Zone"] == label) & df_clean["Start Date"].notna()
    df_clean.loc[m1, "Start_UTC"] = (
        df_clean.loc[m1, "Start Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

    m2 = (df_clean["End Time Zone"] == label) & df_clean["End Date"].notna()
    df_clean.loc[m2, "End_UTC"] = (
        df_clean.loc[m2, "End Date"]
        .dt.tz_localize(tz)
        .dt.tz_convert("UTC")
    )

df_clean["Elapsed_UTC_Minutes"] = (df_clean["End_UTC"] - df_clean["Start_UTC"]).dt.total_seconds() / 60

recorded_minutes = total_duration.loc[df_clean.index].dt.total_seconds() / 60
df_clean["UTC_Duration_Difference_Minutes"] = (df_clean["Elapsed_UTC_Minutes"] - recorded_minutes).abs()

display(
    df_clean[
        ["Start Date", "End Date", "Start_UTC", "End_UTC", "Elapsed_UTC_Minutes", "UTC_Duration_Difference_Minutes"]
    ].head()
)

## 8. Final time-quality filter
We keep only rows that are valid for time-based analysis.

In [ ]:
time_quality_mask = (
    df_clean["Start_UTC"].notna() &
    df_clean["End_UTC"].notna() &
    (df_clean["Elapsed_UTC_Minutes"] > 0) &
    (
        df_clean["UTC_Duration_Difference_Minutes"].isna() |
        (df_clean["UTC_Duration_Difference_Minutes"] <= 2)
    )
)

df_time_clean = df_clean.loc[time_quality_mask].copy()
df_time_review = df_clean.loc[~time_quality_mask].copy()

print("Rows kept for analysis:", len(df_time_clean))
print("Rows sent to review:", len(df_time_review))

## Data Quality Investigation

### 9. Data quality investigation (before clustering)
This section documents problematic records, gives examples, summarizes duration discrepancies, and states cleaning assumptions.

In [ ]:
# 9.1 How many problematic records exist?
problem_flags = pd.DataFrame(index=df_clean.index)

problem_flags["missing_utc"] = df_clean["Start_UTC"].isna() | df_clean["End_UTC"].isna()
problem_flags["nonpositive_elapsed"] = df_clean["Elapsed_UTC_Minutes"] <= 0
problem_flags["large_duration_gap"] = df_clean["UTC_Duration_Difference_Minutes"] > 2
problem_flags["negative_energy"] = df_clean["Energy (kWh)"] < 0

problem_flags["any_problem"] = problem_flags.any(axis=1)

summary_counts = pd.Series({
    "total_rows_in_df_clean": len(df_clean),
    "rows_with_any_problem": int(problem_flags["any_problem"].sum()),
    "rows_missing_utc": int(problem_flags["missing_utc"].sum()),
    "rows_nonpositive_elapsed": int(problem_flags["nonpositive_elapsed"].sum()),
    "rows_duration_gap_gt_2min": int(problem_flags["large_duration_gap"].sum()),
    "rows_negative_energy": int(problem_flags["negative_energy"].sum()),
    "rows_kept_for_modeling_df_time_clean": len(df_time_clean),
    "rows_sent_to_review_df_time_review": len(df_time_review),
})

display(summary_counts.to_frame("count"))

In [ ]:
# 9.2 A few examples of problematic rows
example_cols = [
    "Station Name","User ID","Start Date","End Date",
    "Start Time Zone","End Time Zone",
    "Start_UTC","End_UTC","Elapsed_UTC_Minutes",
    "Total Duration (hh:mm:ss)","UTC_Duration_Difference_Minutes","Energy (kWh)"
]

examples = df_clean.loc[problem_flags["any_problem"], example_cols].copy()

print("Examples (first 15 problematic rows):")
display(examples.head(15))

In [ ]:
# 9.3 Quick summary of duration discrepancies
duration_gap = df_clean["UTC_Duration_Difference_Minutes"]

print("Duration discrepancy summary (minutes):")
display(duration_gap.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("\nTop 10 largest discrepancies:")
display(
    df_clean.sort_values("UTC_Duration_Difference_Minutes", ascending=False)[
        ["Station Name","Start Date","End Date","Total Duration (hh:mm:ss)",
         "Elapsed_UTC_Minutes","UTC_Duration_Difference_Minutes","Energy (kWh)"]
    ].head(10)
)

### Data-quality decisions (report text)

- We keep sessions only when UTC start/end timestamps are available, elapsed UTC duration is positive, and the UTC-vs-recorded duration mismatch is at most 2 minutes.
- We exclude records that fail these checks because they can distort hourly profiles and produce unreliable user behavior features for clustering.
- PST/PDT/UTC handling: we convert each timestamp using its own timezone label (PST, PDT, or UTC) and store a unified UTC timeline to make durations comparable across all records.
- Hourly energy allocation assumption: because only total session energy is known (not second-by-second charging power), we distribute session energy proportionally across the hours overlapped by the session duration.

## 10. Helper function: split one session into hours
We split each charging session into hourly overlaps.

In [ ]:
def split_session_into_hours(start_utc, end_utc):
    rows = []
    h = start_utc.floor("h")

    while h < end_utc:
        h_end = h + pd.Timedelta(hours=1)
        overlap_start = max(start_utc, h)
        overlap_end = min(end_utc, h_end)

        overlap_hours = (overlap_end - overlap_start).total_seconds() / 3600
        if overlap_hours > 0:
            rows.append((h, overlap_hours))

        h = h_end

    return rows

## 11. Build hourly station and user tables
Energy is allocated to each hour proportionally to overlap duration.

In [ ]:
station_rows = []
user_rows = []

for _, r in df_time_clean.iterrows():
    parts = split_session_into_hours(r["Start_UTC"], r["End_UTC"])
    total_h = sum(x[1] for x in parts)

    energy = r["Energy (kWh)"] if pd.notna(r["Energy (kWh)"]) else 0.0

    for hour, occ_h in parts:
        e_h = (energy * occ_h / total_h) if total_h > 0 else 0.0

        station_rows.append({
            "Station Name": r["Station Name"],
            "Hour": hour,
            "Energy (kWh)": e_h,
            "Occupied_Port_Hours": occ_h,
            "Session_Count": 1
        })

        if pd.notna(r["User ID"]) and str(r["User ID"]).strip() != "":
            user_rows.append({
                "User ID": str(r["User ID"]).strip(),
                "Hour": hour,
                "Energy (kWh)": e_h,
                "Occupied_Port_Hours": occ_h,
                "Session_Count": 1
            })

hourly_energy = (
    pd.DataFrame(station_rows)
    .groupby(["Station Name", "Hour"], as_index=False)
    .sum()
)

hourly_user_usage = (
    pd.DataFrame(user_rows)
    .groupby(["User ID", "Hour"], as_index=False)
    .sum()
)

print("hourly_energy rows:", len(hourly_energy))
print("hourly_user_usage rows:", len(hourly_user_usage))
display(hourly_energy.head())

## 12. Add local-time features
We create local hour/day fields for behavior analysis.

In [ ]:
hourly_energy["Hour_Local"] = hourly_energy["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_energy["Hour_Of_Day"] = hourly_energy["Hour_Local"].dt.hour
hourly_energy["Weekday_Local"] = hourly_energy["Hour_Local"].dt.dayofweek

hourly_user_usage["Hour_Local"] = hourly_user_usage["Hour"].dt.tz_convert("America/Los_Angeles")
hourly_user_usage["Hour_Of_Day"] = hourly_user_usage["Hour_Local"].dt.hour
hourly_user_usage["Weekday_Local"] = hourly_user_usage["Hour_Local"].dt.dayofweek

display(hourly_energy.head())
display(hourly_user_usage.head())

## Pre-clustering checks
We verify user coverage, missing values, filter low-activity users, and build a user-level feature table.

1) Check number of users


In [ ]:
n_users = hourly_user_usage["User ID"].nunique()
print("Unique users:", n_users)

display(
    hourly_user_usage["User ID"]
    .value_counts()
    .head(10)
    .rename("hourly_rows_per_user")
)

2) Check missing values in key columns

In [ ]:
key_cols = ["User ID", "Hour", "Energy (kWh)", "Occupied_Port_Hours"]
print("Missing values in key columns:")
display(hourly_user_usage[key_cols].isna().sum())

# keep only valid key rows
hourly_user_usage = hourly_user_usage.dropna(subset=key_cols).copy()
print("Rows after key-column NA drop:", len(hourly_user_usage))

3) Filter very low-activity users

In [ ]:
# define activity as number of active hour-rows per user
user_activity = (
    hourly_user_usage.groupby("User ID")
    .size()
    .rename("active_hour_rows")
    .reset_index()
)

min_rows = 24 
active_users = user_activity[user_activity["active_hour_rows"] >= min_rows]["User ID"]

hourly_user_usage_f = hourly_user_usage[
    hourly_user_usage["User ID"].isin(active_users)
].copy()

print("Users before filter:", hourly_user_usage["User ID"].nunique())
print("Users after filter:", hourly_user_usage_f["User ID"].nunique())
print("Rows after filter:", len(hourly_user_usage_f))

4) Buid user-feature table (one row = one user)

In [ ]:
# total behavior features
base_features = (
    hourly_user_usage_f.groupby("User ID")
    .agg(
        total_energy_kwh=("Energy (kWh)", "sum"),
        avg_energy_per_active_hour=("Energy (kWh)", "mean"),
        total_occupied_hours=("Occupied_Port_Hours", "sum"),
        avg_occupied_per_active_hour=("Occupied_Port_Hours", "mean"),
        active_hour_rows=("Hour", "count")
    )
)

# hour-of-day behavior profile (0..23)
hour_profile = (
    hourly_user_usage_f.pivot_table(
        index="User ID",
        columns="Hour_Of_Day",
        values="Energy (kWh)",
        aggfunc="sum",
        fill_value=0
    )
)

# normalize to shares (so we cluster behavior pattern, not only volume)
hour_profile = hour_profile.div(hour_profile.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
hour_profile.columns = [f"share_h{int(c):02d}" for c in hour_profile.columns]

# weekday/weekend preference
tmp = hourly_user_usage_f.copy()
tmp["is_weekend"] = tmp["Weekday_Local"].isin([5, 6])

week_profile = (
    tmp.groupby(["User ID", "is_weekend"])["Energy (kWh)"]
    .sum()
    .unstack(fill_value=0)
)

week_profile.columns = ["weekday_energy", "weekend_energy"] if False in week_profile.columns and True in week_profile.columns else week_profile.columns
if "weekday_energy" not in week_profile.columns:
    week_profile["weekday_energy"] = 0.0
if "weekend_energy" not in week_profile.columns:
    week_profile["weekend_energy"] = 0.0

week_total = (week_profile["weekday_energy"] + week_profile["weekend_energy"]).replace(0, np.nan)
week_profile["weekday_share"] = (week_profile["weekday_energy"] / week_total).fillna(0)
week_profile["weekend_share"] = (week_profile["weekend_energy"] / week_total).fillna(0)

# final clustering table
user_features = base_features.join(hour_profile, how="left").join(
    week_profile[["weekday_share", "weekend_share"]], how="left"
).fillna(0)

print("user_features shape:", user_features.shape)
display(
    user_features
    .sort_values("User ID", ascending=True)
    .head(10)
)